# Parallel Computation

This notebook describes how to use LightCurveLynx to perform parallel computation. 

The core simulation function of the model can take a `concurrent.futures.Executor` object and use that to distribute the computation over multiple processes. This object can be a built in parallelization method, such as `ThreadPoolExecutor` or `ProcessPoolExecutor`, or other libraries, such as Dask.

Note:
  * By default, every batch sent to a worker carries its own full copy of the model and survey data. The [Preloading worker data](#Preloading-worker-data) section below describes `make_worker_initializer`, which sends that data to each worker once instead.
  * In some cases, the default parallelization approach may fail with a `PickleError` or something like `AttributeError: Can't get attribute 'XXX' on <module '__main__' (built-in)>`. In such cases, we recommend using a third-party library such as Loky, Dask, or Ray; see the examples below.

In [ ]:
from lightcurvelynx.astro_utils.passbands import PassbandGroup
from lightcurvelynx.models.basic_models import ConstantSEDModel
from lightcurvelynx.obstable.opsim import OpSim
from lightcurvelynx.simulate import WorkerPreloadWarning, make_worker_initializer, simulate_lightcurves
from lightcurvelynx.survey_info import SurveyInfo

# Usually we would not hardcode the path to the passband files, but for this demo we will use a relative path
# to the test data directory so that we do not have to download the files.
table_dir = "../../tests/lightcurvelynx/data/passbands"

## Prerequisite Data

We start by loading the standard information that we need for any simulation.

  * An `ObsTable` that includes the survey’s pointing and noise information.
  * A `PassbandGroup` for that survey.

We start by creating a toy survey that includes pointings at two locations (0.0, 10.0) and (180.0, -10.0) in the "g" and "r" bands and loading the passband group.

As of v0.4.0, we wrap these data structures in a ``SurveyInfo`` object.

In [ ]:
obsdata1 = {
    "time": [0.0, 1.0, 2.0, 3.0],
    "ra": [0.0, 0.0, 180.0, 180.0],
    "dec": [10.0, 10.0, -10.0, -10.0],
    "filter": ["g", "r", "g", "r"],
    "zp": [5.0, 6.0, 7.0, 8.0],
    "seeing": [1.12, 1.12, 1.12, 1.12],
    "skybrightness": [20.0, 20.0, 20.0, 20.0],
    "exptime": [29.2, 29.2, 29.2, 29.2],
    "nexposure": [2, 2, 2, 2],
}
obstable1 = OpSim(obsdata1)

passband_group1 = PassbandGroup.from_preset(
    preset="LSST",
    table_dir=table_dir,
    filters=["g", "r", "i"],
)

survey_info1 = SurveyInfo(obstable=obstable1, passbands=passband_group1)

## Model Creation

Next we create a model from which to simulate observations. We define a model and its parameters as we would with any other simulation.  Here we use a constant SED model (same value for all times and wavelengths). We place the object at (0.0, 10.0) so it is observed by some of the pointings from each survey.

In [ ]:
model = ConstantSEDModel(brightness=100.0, t0=0.0, ra=0.0, dec=10.0, redshift=0.0, node_label="my_star")

## Preloading worker data

By default, every batch sent to a worker carries a full copy of the model and the survey data (every `ObsTable`, `PassbandGroup` and noise model it depends on). For the toy survey above that costs very little, but for the full LSST OpSim each `ObsTable` alone is close to a gigabyte. Sending that gigabyte with every batch is slow: a run of 25 batches on 2 workers that takes about 0.3 s serially can take about 90 s because of the per-batch copies, before the workers have done any simulation work.

`make_worker_initializer(model, survey_info)` avoids this. It serializes the model and survey data once and returns a picklable, zero-argument callable (an *initializer*) that installs that data on a worker the first time it runs there. Pass it to your backend's per-worker setup hook:

| Backend | Usage |
|---|---|
| `ProcessPoolExecutor` | `ProcessPoolExecutor(2, initializer=initializer)` |
| `ThreadPoolExecutor` | `ThreadPoolExecutor(2, initializer=initializer)` |
| `multiprocessing.Pool` | `multiprocessing.Pool(2, initializer)` |
| loky | `loky.get_reusable_executor(max_workers=2, initializer=initializer)` |
| ray | `ray.util.multiprocessing.Pool(2, initializer=initializer)` |
| dask | `client.register_worker_callbacks(initializer)`, then `executor=client` |

Once a backend's workers are preloaded, `simulate_lightcurves` sends each batch without the model or survey data, and the worker uses the data it already has. Passing `num_jobs=` instead of `executor=` does this automatically: it builds the initializer and the `ProcessPoolExecutor` for you, so there is nothing extra to do.

**Freshness.** Before sending any batches, `simulate_lightcurves` computes a content hash of the current model and survey data and compares it to what was preloaded. This costs about 2 s for the full OpSim, but only once per call, not once per batch. If the data doesn't match, the run falls back to sending a full copy of the model and survey data with every batch and emits a `WorkerPreloadWarning` saying why. This matters because running a model -- even serially -- can change its internal state (for example, `TableSampler` tracks which rows it has already handed out), which changes what gets pickled even though nothing about the model's configuration changed. So create the initializer *after* the last time you use the model serially, right before the parallel run.

**`num_jobs=` and pickling.** `num_jobs=` preloads its pool with the standard library's `pickle`, the same way a plain `ProcessPoolExecutor` pickles its batches, so it behaves exactly as it did before preloading existed for anything defined in your script: functions and classes in `__main__` are pickled by reference. `make_worker_initializer` uses `cloudpickle` instead when it is installed, which is what lets loky, dask and ray run classes defined in a notebook. A `num_jobs=` run leaves nothing registered in this process, so a later run with a plain `executor=` that was never preloaded still warns that "The executor's workers were not preloaded with make_worker_initializer()." Only if you called `make_worker_initializer` yourself for the same model and survey data does such a run warn instead that "N of N batches reached workers that were not preloaded with this model and survey data." Both warnings mean the same thing: preload that executor too, with `initializer=make_worker_initializer(model, survey_info)`.

**Thread safety.** Each batch sent to a preloaded worker gets its own freshly unpickled copy of the model, so models that keep per-call state on `self` (for example `SncosmoWrapperModel`, which calls `self.source.set(...)` then `self.source.flux(...)`) are safe to run on a `ThreadPoolExecutor` or dask threads. This is *not* true of batches that fall back to the old path: those share a single model object across every thread in one process, which is unsafe for the same kind of model.

**Dask scheduler memory.** The dask scheduler keeps a copy of every callback registered with `register_worker_callbacks` so that it can also run it on workers that join the cluster later. Registering again with a new initializer adds another copy rather than replacing the old one, so a long-lived session that preloads many different datasets grows the scheduler's memory by roughly one payload (up to ~0.9 GB for the full OpSim) per registration. Workers themselves always keep only the most recently installed preload, so results stay correct either way. `client.run(initializer)` runs the initializer immediately without this build-up, but it only reaches the workers that already exist, not ones that join afterward.

## Simulation

We build the initializer right after creating the model (and before using the model any other way), then pass it to a `ProcessPoolExecutor`:

In [ ]:
import concurrent.futures

initializer = make_worker_initializer(model, survey_info1)

with concurrent.futures.ProcessPoolExecutor(max_workers=4, initializer=initializer) as executor:
    results = simulate_lightcurves(
        model=model,
        num_samples=10_000,
        survey_info=survey_info1,
        obstable_save_cols=["zp_nJy"],
        executor=executor,
        batch_size=100,
    )

print(f"Generated {len(results)} light curves")
print(results["lightcurve"][0])

If we do not provide an executor object, but rather a number of jobs, we automatically create and manage the `ProcessPoolExecutor`. This also preloads its workers for us, so there is no initializer to build by hand (see the note on `num_jobs=` and pickling above). Here we run the simulation on 4 processes.

In [ ]:
results = simulate_lightcurves(
    model=model,
    num_samples=10_000,
    survey_info=survey_info1,
    obstable_save_cols=["zp_nJy"],
    num_jobs=4,
    batch_size=100,
)

print(f"Generated {len(results)} light curves")
print(results["lightcurve"][0])

## ThreadPoolExecutor

Threads share memory within one process, so the main benefit of preloading a `ThreadPoolExecutor` is the fresh-model-per-batch guarantee from the Thread safety note above: it lets models with per-call state (like `SncosmoWrapperModel`) run safely on a thread pool. The model and survey data haven't changed since we built `initializer` above, so we can reuse it directly.

In [ ]:
with concurrent.futures.ThreadPoolExecutor(max_workers=4, initializer=initializer) as executor:
    results = simulate_lightcurves(
        model=model,
        num_samples=10_000,
        survey_info=survey_info1,
        obstable_save_cols=["zp_nJy"],
        executor=executor,
        batch_size=100,
    )

print(f"Generated {len(results)} light curves")
print(results["lightcurve"][0])

## multiprocessing.Pool

The standard library's `multiprocessing.Pool` also accepts an initializer, passed positionally.

In [ ]:
import multiprocessing

with multiprocessing.Pool(4, initializer) as executor:
    results = simulate_lightcurves(
        model=model,
        num_samples=10_000,
        survey_info=survey_info1,
        obstable_save_cols=["zp_nJy"],
        executor=executor,
        batch_size=100,
    )

print(f"Generated {len(results)} light curves")
print(results["lightcurve"][0])

## Loky

[Loky](https://loky.readthedocs.io/) is a process-based parallelization library used by `joblib`.
We recommend it when having pickle errors with the default executor (`ProcessPoolExecutor`). Loky uses `cloudpickle`, which also keeps classes defined in a notebook's `__main__` working; `make_worker_initializer` uses `cloudpickle` too, when it is installed, for the same reason.

**Note:** Loky is not installed by default, so users will need to install loky (`pip install 'loky'`) to run this cell.

In [ ]:
try:
    import loky

    executor = loky.get_reusable_executor(max_workers=4, initializer=initializer)
    results = simulate_lightcurves(
        model=model,
        num_samples=100,
        survey_info=survey_info1,
        obstable_save_cols=["zp_nJy"],
        executor=executor,
    )
    print(f"Generated {len(results)} light curves")
    print(results["lightcurve"][0])
except ImportError:
    print("Loky is not installed, skipping Loky example")

## Dask

[Dask](https://dask.org) is a framework for parallel task execution at scale.
We recommend Dask for a web-based dashboard to monitor the resource utilization.
Dask also supports distributed cluster runs via [dask-kubernetes](https://kubernetes.dask.org/en/latest/) and [dask-jobqueue](https://jobqueue.dask.org/en/latest/) packages.

Here we provide an example for a simple local cluster, preloading its workers with `client.register_worker_callbacks(initializer)` before running the simulation. Note the scheduler-memory caveat for this call in the Preloading worker data section above. Like loky, Dask uses `cloudpickle`, so classes defined in this notebook's `__main__` work with it too.

**Note:** Dask is not installed by default, so users will need to install dask (`pip install 'dask[distributed]'`) to run this cell.
You would also need `bokeh` package for the dashboard (`pip install bokeh`).

In [ ]:
try:
    import dask.distributed

    with dask.distributed.Client() as client:
        print(f"Dask dashboard link: {client.dashboard_link}")
        client.register_worker_callbacks(initializer)
        results = simulate_lightcurves(
            model=model,
            num_samples=100,
            survey_info=survey_info1,
            obstable_save_cols=["zp_nJy"],
            executor=client,
        )
    print(f"Generated {len(results)} light curves")
    print(results["lightcurve"][0])
except ImportError:
    print("Dask is not installed, skipping Dask example")

## Ray

[Ray](https://ray.io) is a framework for distributed application execution.

We can parallelize the computation via Ray by using `ray.util.multiprocessing.Pool`, passing the initializer the same way as with `multiprocessing.Pool`. Like loky and Dask, Ray uses `cloudpickle`, so classes defined in this notebook's `__main__` work with it too.

**Note:** Ray is not installed by default, so users will need to install ray (`pip install -U "ray[default]"`) to run this cell.

In [ ]:
try:
    import ray
    from ray.util.multiprocessing import Pool

    with Pool(processes=4, initializer=initializer) as executor:
        results = simulate_lightcurves(
            model=model,
            num_samples=100,
            survey_info=survey_info1,
            obstable_save_cols=["zp_nJy"],
            executor=executor,
        )
    print(f"Generated {len(results)} light curves")
    print(results["lightcurve"][0])

    ray.shutdown()
except ImportError:
    print("Ray is not installed, skipping Ray example")

## Saving to Files

Depending on the size of the simulated results, you might not want to load the full set into memory as a single table. The `simulate_lightcurves` has a function to save each shard (the result of each process) to a unique file. Instead of returning the NestedFrames, the function returns the list of file paths containing the data. Users can then analyze or load these later.

In [ ]:
file_paths = simulate_lightcurves(
    model=model,
    num_samples=10_000,
    survey_info=survey_info1,
    num_jobs=4,
    batch_size=1000,
    obstable_save_cols=["zp_nJy"],
    output_file_path="./scratch/nb_results.parquet",
)
print(file_paths)

As you can see the results are broken up into ten different files.

## Handling Randomness

By default LightCurveLynx creates a new random number generator (without a fixed seed) so that the parameters will vary from run to run. However it also all allows the users to control the randomness by passing in their own random number generator, which may have a fixed seed. In both these cases the ``simulate_lightcurves()`` ensures the correct behavior during parallel runs.

The initial random number generator (provided or default) is used to create a new random seed for each processing shard. If the user had provided a seeded random number generator, the new list of seeds will be predefined. Thus each shard will get a predefined (but different) random number generator. If the initial random number generator was not seeded, the seeds for each shard will themselves vary from run to run.

## Working with Fixed Sets of Data

Several of the sampling nodes, such as the `TableSampler`, `GivenValueSampler`, or `GivenValueList` work with predefined lists of data. If the data is drawn randomly from the lists, such as with the `GivenValueSampler`, this randomness will be handled as described above. Each shard will use a different random seed to start from a different part of the sampling space.

In contrast there are a few nodes whose behavior is designed to be deterministic:
  * `GivenValueList` is meant for testing only and will fail if you try to use it in a parallel run.
  * `TableSampler` (with `in_order=True`) automatically handles the coordination so that each worker uses a disjoint range of rows. For example, if we are sampling 100 values using batches of size 10, the first shard will use rows [0, 9]. The second shard will use rows [10, 19]. And so forth.

In [ ]:
from lightcurvelynx.math_nodes.given_sampler import TableSampler

table_data = {
    "ra": [1, 2, 3, 4, 5, 6, 7, 8, 9, 10],
    "dec": [10, 9, 8, 7, 6, 5, 4, 3, 2, 1],
}
table_node = TableSampler(table_data, in_order=True, node_label="table_node")

model2 = ConstantSEDModel(
    brightness=100.0,
    t0=0.0,
    ra=table_node.ra,
    dec=table_node.dec,
    redshift=0.0,
    node_label="my_star",
)

# TableSampler changes its own internal state as it hands out rows, so we build this model's
# initializer right away and never use model2 serially before the parallel run below.
initializer2 = make_worker_initializer(model2, survey_info1)

with concurrent.futures.ProcessPoolExecutor(max_workers=4, initializer=initializer2) as executor:
    results = simulate_lightcurves(
        model=model2,
        num_samples=5,
        survey_info=survey_info1,
        executor=executor,
        batch_size=2,
    )

for idx in range(5):
    print(f"{idx}: RA={results['ra'][idx]}, DEC={results['dec'][idx]}")

## Overhead

As with any distributed computation, there is per-call and per-batch overhead from pickling and unpickling data. Preloading moves most of that cost out of the per-batch path, leaving:

  * **One serialization**, done once by `make_worker_initializer`, no matter how many batches or workers follow.
  * **One freshness check per `simulate_lightcurves` call**: a content digest of the current model and survey data, computed without keeping the pickled bytes. This costs about 2 s for the full LSST OpSim, regardless of the number of batches.
  * **One unpickle per worker of the shared survey data** (every `SurveyInfo` and any `ObsTable` reachable from the model), the first time that worker handles a batch, instead of once per batch.

Preloading does *not* remove all per-batch unpickling, though:

  * **Each batch still unpickles its own small copy of the model.** This is deliberate: it is what keeps preloaded batches thread-safe, since no two batches ever share a model object (see Thread safety above).
  * **Large non-`ObsTable` data inside the model** -- a big `TableSampler` catalog, a template library, a dust map -- is still unpickled once per batch, exactly as it was before preloading. Preloading only removes the parent-side pickling and transfer of the model and shared survey data; it does not shrink what's inside the model itself.

The user can provide a `batch_size` parameter to control the target batch size for each process, to make sure each batch has enough work to be worth sending.